In [1]:
import pandas as pd
import numpy as np

# 1. Load Dataset
print("=== CAPSTONE DATASET QUALITY AUDIT ===")
file_path = "capstone_2023_merged.csv"
df = pd.read_csv(file_path, low_memory=False)

print(f"\n1. Shape Audit:")
print(f"   - Total Observations (Rows): {df.shape[0]:,}")
print(f"   - Total Features (Columns):  {df.shape[1]}")

# 2. Key Target & Feature Missingness Check
target_col = 'amountPaidOnBuildingClaim' if 'amountPaidOnBuildingClaim' in df.columns else 'amountPaidOnTotalClaim'
core_cols = [target_col, 'DAMAGE_PROPERTY', 'primary_landcover_code', 'alternate_landcover_code']
core_cols = [c for c in core_cols if c in df.columns]

print(f"\n2. Core Attributes Missing Value Check:")
for col in core_cols:
    missing_count = df[col].isna().sum()
    missing_pct = (missing_count / len(df)) * 100
    print(f"   - {col:30s}: {missing_count:,} missing ({missing_pct:.2f}%)")

# 3. Alphanumeric Parse Validation (NOAA Property Damage)
def parse_damage(val):
    if pd.isna(val): return 0.0
    val_str = str(val).strip().upper()
    if val_str.endswith('K'): return float(val_str[:-1]) * 1_000
    elif val_str.endswith('M'): return float(val_str[:-1]) * 1_000_000
    elif val_str.endswith('B'): return float(val_str[:-1]) * 1_000_000_000
    try: return float(val_str)
    except ValueError: return 0.0

if 'DAMAGE_PROPERTY' in df.columns:
    df['DAMAGE_PROPERTY_NUMERIC'] = df['DAMAGE_PROPERTY'].apply(parse_damage)
    print(f"\n3. NOAA Damage Parsing Validation:")
    print(f"   - Zero/Unspecified Damage Records: {(df['DAMAGE_PROPERTY_NUMERIC'] == 0).sum():,}")
    print(f"   - Non-Zero Damage Records:         {(df['DAMAGE_PROPERTY_NUMERIC'] > 0).sum():,}")
    print(f"   - Max Property Damage Parsed:      ${df['DAMAGE_PROPERTY_NUMERIC'].max():,.2f}")

# 4. Binary Classification Balance Check ($10k Threshold)
df['payout_numeric'] = pd.to_numeric(df[target_col], errors='coerce').fillna(0)
high_impact_count = (df['payout_numeric'] >= 10000).sum()
low_impact_count = (df['payout_numeric'] < 10000).sum()

print(f"\n4. Target Distribution ($10k Threshold):")
print(f"   - Low/Zero Payout (<$10k):  {low_impact_count:,} ({low_impact_count/len(df)*100:.2f}%)")
print(f"   - High Impact Payout (≥$10k): {high_impact_count:,} ({high_impact_count/len(df)*100:.2f}%)")

print("\n=== AUDIT COMPLETE: DATA PRESERVED & CLEAN ===")

=== CAPSTONE DATASET QUALITY AUDIT ===

1. Shape Audit:
   - Total Observations (Rows): 88,195
   - Total Features (Columns):  138

2. Core Attributes Missing Value Check:
   - amountPaidOnBuildingClaim     : 6,716 missing (7.61%)
   - DAMAGE_PROPERTY               : 8 missing (0.01%)
   - primary_landcover_code        : 0 missing (0.00%)
   - alternate_landcover_code      : 0 missing (0.00%)

3. NOAA Damage Parsing Validation:
   - Zero/Unspecified Damage Records: 75,549
   - Non-Zero Damage Records:         12,646
   - Max Property Damage Parsed:      $500,000,000.00

4. Target Distribution ($10k Threshold):
   - Low/Zero Payout (<$10k):  39,452 (44.73%)
   - High Impact Payout (≥$10k): 48,743 (55.27%)

=== AUDIT COMPLETE: DATA PRESERVED & CLEAN ===
